# 研究架構

::: {.callout-tip}

投影片操作：**Alt + 點擊** 可縮放任何圖片/表格（reveal.js zoom）；`O` 鍵總覽、`F` 全螢幕。

:::

本研究檢驗配對交易兩個環節能否以機器學習與深度學習改良：

| | 命題 | 對照設計 | 結果 |
| :--- | :--- | :--- | :--- |
| **命題 1** | **機器學習分群**能建立優於傳統產業分類的配對搜尋空間 | 4 分組 × 3 排序矩陣（分組為唯一變因） | **未獲支持** |
| **命題 2** | **深度強化學習交易**優於傳統 Z-Score 規則 | 5 種配對底 × 交易端（交易端為唯一變因） | **獲得支持** |

**共同控制條件**：混合特徵（報酬 PCA ⊕ PIT 基本面 ⊕ GICS one-hot）、
共整合篩選（ADF 0.05 + OU 半衰期 + Hurst）、S&P 500 歷史成分股 2000–2025、
形成期 252 日 / 交易期 126 日 / 滾動 21 日、交易成本單邊 0.29%。

::: {.callout-important}

### 本研究的推論口徑

所有跨策略比較均以**同參數網格逐格配對檢定**為準，而非比較各自的網格最佳值。
15 格中的最大值會系統性地納入運氣成分——本研究有兩處結論在改用配對檢定後被推翻
（命題 1 的方向、以及命題 2 中各配對底的排名）。

:::


# 方法論：形成期四層架構

策略 = 四個可獨立替換的層之組態，由中性組裝器依參數組裝：

| 層 | 職責 | 本研究採用的選項 |
| :--- | :--- | :--- |
| **特徵** | 個股 → 特徵向量 | 報酬 PCA 因子載荷 ⊕ 基本面 ⊕ 產業 one-hot |
| **分組** | 特徵 → 配對搜尋空間 | GICS 產業／HDBSCAN／Agglomerative／K-means |
| **排序** | 群內配對 → 優先序 | SSD／DTW／SSD-DTW-PCA |
| **篩選** | 統計檢定淘汰 | ADF 共整合 + OU 半衰期 + Hurst |

交易期則為 **Z-Score 狀態機**（規則型基準）或 **DRL 門檻選擇式**（Kim & Kim 2019 風格）。

> 此架構使「分組」「排序」「交易端」皆可作為單變因替換，是兩大命題能乾淨對照的前提。
> 實作正確性以數值回歸測試保證：組裝器復現原生策略的結果為**逐位元相同**。


# 參考文獻與方法對應

## 分群方法
> Campello, Moulavi & Sander (2013). Density-based clustering based on hierarchical density estimates. *PAKDD*.
> Ward (1963). Hierarchical grouping to optimize an objective function. *JASA*, **58**(301).
> MacQueen (1967). Some methods for classification and analysis of multivariate observations.

HDBSCAN（自動群數、噪音標記）、Agglomerative（dendrogram 分位數校準）、
K-means（群數對齊同期 Agglomerative，使量級可比）。

## 排序準則
> Gatev, Goetzmann & Rouwenhorst (2006). Pairs trading. *RFS*, **19**(3).　📄 `ref/2006-...pdf`
> 許鈞翔 (2025)。最小距離法結合動態時間校正之配對交易研究。　📄 `ref/2025-...pdf`

SSD（同步距離）、DTW（Sakoe-Chiba 時間扭曲）、SSD-DTW-PCA（兩距離 PCA 融合）。

## 特徵與交易端
> Avellaneda & Lee (2010). Statistical arbitrage in the U.S. equities market. *Quantitative Finance*, **10**(7).　📄
> Hong & Hwang (2021). In search of pairs using firm fundamentals. *EJF*, **29**(5).　📄
> Kim & Kim (2019). Optimizing the pairs-trading strategy using DRL with trading and stop-loss boundaries. *Complexity*.　📄

報酬 PCA 因子載荷（Avellaneda & Lee）、基本面配對（Hong & Hwang）、
DRL 門檻選擇式交易（Kim & Kim）。


# 命題 1：機器學習分群 vs 傳統產業分類

**設計**：固定特徵、篩選、Z-Score 交易端；變動分組方法 × 排序準則。

## 各格網格最佳值（最佳年化 / 最佳 Sharpe）

| 分組＼排序 | SSD | DTW | SSD-DTW-PCA |
| :--- | :---: | :---: | :---: |
| **GICS（傳統）** | 1.66% / 0.20 | 1.10% / 0.18 | 1.65% / **0.35** |
| **HDBSCAN** | 1.29% / 0.17 | 0.89% / 0.15 | **1.70%** / 0.23 |
| **Agglomerative** | **1.77% / 0.26** | 0.60% / 0.11 | 1.13% / 0.18 |
| **K-means** | 1.31% / 0.20 | −0.06% / 0.03 | 0.67% / 0.12 |

若僅看此表，會得到「Agglomerative × SSD 優於 GICS × SSD（1.77% vs 1.66%）」的印象。
然而兩者皆為 15 格中的最大值，該差距未經檢定。


## 命題 1 的配對檢定：結論與上表相反

同排序、同參數網格逐格配對，$n = 15$，$H_0$：ML 分群與 GICS 分組績效相同。

| 分組＼排序 | SSD | DTW | SSD-DTW-PCA |
| :--- | :---: | :---: | :---: |
| HDBSCAN | −0.042 (**p=0.021**) | −0.057 (p=0.057) | −0.073 (**p=0.027**) |
| Agglomerative | +0.021 (p=0.586) | −0.177 (**p<0.001**) | −0.183 (**p<0.001**) |
| K-means | −0.038 (p=0.302) | −0.246 (**p<0.001**) | −0.285 (**p<0.001**) |

*ΔSharpe（ML − GICS），粗體為 5% 顯著；以年化報酬複核結論相同*

**9 組比較中，5 組顯著劣於 GICS，0 組顯著優於。**
最佳情形（Agglomerative × SSD）僅為「與 GICS 無顯著差異」（勝 9/15 格）。

> **命題 1 未獲支持**：在本研究設定下，機器學習分群未能建立優於 GICS 產業分類的
> 配對搜尋空間。上一頁的表格差異源於各自取網格最大值，屬選擇偏誤。


## 命題 1 為何失敗：兩項結構性代價

ML 分群與 GICS 分組的差異可完全歸結為兩項可測量的結構變化：

| 分組法 | 跨產業配對比例 | 期均配對數 |
| :--- | :---: | :---: |
| **GICS（傳統）** | **0.0%**（定義上不可能） | 15.0 |
| HDBSCAN | **24.9%** | 15.5 |
| Agglomerative | 10.7% | 11.5 |
| K-means | 12.3% | **8.7** |

**代價一：破壞產業一致性。** ML 分群依特徵空間的幾何鄰近性分組，
會將不同產業但特徵相近的股票歸為同群，產生 11–25% 的跨產業配對。
產業一致性是配對交易均值回歸假設的經濟基礎（共同的需求衝擊、成本結構、監管環境）——
跨產業配對的價差即使歷史上接近，也缺乏使其回歸的經濟機制。
跨產業比例最低的 Agglomerative（10.7%）正是唯一與 GICS 無顯著差異者。

**代價二：縮減候選池。** 分群將母體切成數十個小群，
群內可列舉的配對數遠少於 GICS 的 11 個大產業。
K-means 期均僅能選出 8.7 對（目標 20 對），無法填滿——
排序準則被迫接受品質更差的候選。

> 此機制與本研究另一項發現同源：收緊 ADF 門檻至 0.01 亦使績效下降
> （1.77% → 1.27%），因為更嚴的篩選同樣迫使排序往距離更遠的候選尋找。
> **兩者共同指向：候選池的充足程度是配對品質的關鍵限制因素。**


## 粒度掃描：分群演算法從來不是關鍵變因

前頁的兩項代價在原設計中互相混淆——三種分群演算法產生的群數本就不同，
無法斷定落後源於「演算法」還是「粒度」。

**受控實驗**：固定 Agglomerative 演算法、特徵、SSD 排序、篩選與交易端，
唯一變因為切割門檻分位 $q$（分位越高 → 群越少越大 → 候選池越大）。

| 門檻分位 | 期均配對數 | 跨產業% | 最佳年化 | 網格均 Sharpe | vs GICS $p$ |
| :---: | :---: | :---: | :---: | :---: | :---: |
| 50 | 6.2 | 9.7% | −0.13% | −0.237 | 0.520 |
| **60** | 8.7 | 10.3% | **1.92%** | **−0.176** | 0.142 |
| 75（基準） | 11.5 | 10.7% | 1.77% | −0.240 | 0.355 |
| 90 | **15.0** | 31.9% | 0.40% | −0.297 | 0.248 |
| 95 | 16.8 | **50.0%** | 0.38% | −0.249 | 0.430 |
| **GICS** | **15.0** | **0.0%** | 1.66% | −0.261 | — |

**發現一：粒度的影響遠大於演算法的影響。**
單一參數即使最佳年化自 −0.13% 變動至 1.92%（幅度 2.05pp）；
相較之下，固定粒度下三種分群演算法的差距僅 0.48pp（AGG 1.77 / KM 1.31 / HDB 1.29）。
**命題 1 原本比較的「分群演算法」，並非決定配對品質的主要變因。**

**發現二：關係為倒 U 形，非單調。** 峰值在 $q$ = 60–75，兩端皆劣化。
兩個機制反向作用——
粒度過細（$q$=50）候選池僅 6.2 對，填不滿目標而被迫接受劣質配對；
粒度過粗（$q$≥90）池雖變大，但跨產業比例自 10% 暴增至 32–50%。
候選池大小與跨產業比例的 Spearman $ho = +1.000$：**兩者在 ML 分群下無法解耦。**


## GICS 為何勝出：它同時取得兩項優勢

::: {.callout-important}

### 一組自然的受控對照

門檻分位 90 的 ML 分群與 GICS 產業分組**期均配對數同為 15.0**——
候選池大小完全相同，唯一差異是跨產業比例（31.9% vs 0.0%）。

該組對照下 GICS 較優（Δ年化 −0.148pp、ΔSharpe −0.036，ML 僅勝 4/15 格），
惟未達統計顯著（$p$ = 0.14 ~ 0.25）。

:::

產業分組之所以難以被超越，在於它**同時**取得兩項在 ML 分群下互斥的性質：

| | 候選池大小 | 產業一致性 |
| :--- | :---: | :---: |
| ML 分群（細） | ✘ 不足 | ✔ 高 |
| ML 分群（粗） | ✔ 充足 | ✘ 被破壞 |
| **GICS** | **✔ 充足** | **✔ 完全一致（定義上）** |

ML 分群依特徵空間的幾何鄰近性切割，擴大群體必然混入其他產業的股票；
GICS 則以外生的產業定義直接給出「大且純」的分組。

**這是命題 1 未獲支持的根本原因**，也界定了 ML 分群可能勝出的條件：
唯有當特徵空間能表達產業以外、且與均值回歸相關的相似性維度時，
跨產業配對的代價才可能被補償。本研究的特徵消融
（動量、結構性財報比率，見附錄 C）未能找到此種維度。

> **誠實界定**：各粒度層級與 GICS 的配對檢定皆未達顯著（$p$ = 0.14 ~ 0.52），
> 故本節結論為「ML 分群未展現優勢且機制可解釋」，
> 而非「GICS 在統計上顯著較優」。


# 命題 2：深度強化學習交易 vs Z-Score

**設計**：**固定形成期配對**，僅將交易端由 Z-Score 換成 DRL 門檻選擇式。
配對底涵蓋三種 ML 分群與**傳統 GICS 分組**，以檢驗增益是否依賴特定配對來源。

同參數網格逐格配對檢定（$n = 15$）：

| 配對底 | ΔSharpe | 勝格數 | $p$ | 最佳年化（Z → DRL） |
| :--- | :---: | :---: | :---: | :---: |
| **GICS × SSD（傳統）** | **+0.402** | 13/15 | 0.0013 | 1.66% → 2.13% |
| HDBSCAN × SDP | +0.317 | 13/15 | 0.0037 | 1.70% → 2.50% |
| GICS × SDP（傳統） | +0.281 | 12/15 | 0.0070 | 1.65% → 1.97% |
| K-means × SSD | +0.285 | 14/15 | 0.0017 | 1.31% → 1.90% |
| Agglomerative × SSD | +0.263 | **15/15** | 0.0006 | 1.77% → 2.37% |

**五種配對底全部顯著改善**（$p < 0.01$）。
增益最大者為傳統 GICS 分組——**DRL 的價值不依賴機器學習分群**。

**穩健性**（15 格中 Sharpe 為正者）：

| 配對底 | Z-Score | → DRL |
| :--- | :---: | :---: |
| GICS × SSD | 5/15 | **13/15** |
| HDBSCAN × SDP | 4/15 | 8/15 |
| Agglomerative × SSD | 3/15 | 8/15 |
| K-means × SSD | 2/15 | 5/15 |


## 命題 2 的三個發現

**1. 增益普適於所有配對底，包含傳統分組**
五種配對底（3 種 ML 分群 + 2 種 GICS 排序組合）疊加 DRL 後全部顯著改善，
$p$ 值介於 0.0006 ~ 0.0070。**傳統 GICS 底的增益最大（+0.402）**，
顯示 DRL 交易端是一個與配對來源正交的獨立改良——
此普適性比僅在 ML 配對底上驗證更強。

**2. DRL 大幅提升參數穩健性**
GICS × SSD 底的正 Sharpe 網格數自 5/15 增至 13/15。
DRL 為每組配對每期自選門檻，等同於將原本需人工調校的參數交由模型依情境決定，
因而降低對外生參數設定的敏感度。

**3. 增益來自門檻選擇而非拒絕交易（見下節行為解析）**
DRL 的動作選單含 SKIP，但損益分解顯示 SKIP 僅貢獻 31–52%，
且被跳過的配對與留下的配對在虧損比例上無顯著差異。
主要增益（48–69%）來自為留下的配對選擇非基準門檻。

> **DRL 未固定隨機種子**，上表為單次訓練值。三種 ML 配對底的五輪重訓結果見下節。


## 命題 2 的重訓穩健性（五輪獨立訓練）

DRL 網路未固定隨機種子，每輪重新訓練。下表為五輪各自取網格最佳後的跨輪統計。

| 配對底 | 最佳年化 中位［範圍］ | 最佳 Sharpe 中位［範圍］ | 正 Sharpe 最少 |
| :--- | :---: | :---: | :---: |
| Agglomerative × SSD | 2.39%［2.34, 2.65］ | 0.34［0.34, 0.38］ | 6/15 |
| HDBSCAN × SSD-DTW-PCA | **2.46%**［2.41, 2.54］ | 0.31［0.31, 0.32］ | **8/15** |
| K-means × SSD | 1.73%［1.53, 1.90］ | 0.26［0.23, 0.28］ | 5/15 |

::: {.callout-important}

### 為何必須報告變異數

Agglomerative 底的單次訓練值為 **2.65%**，恰為其五輪範圍的**上界**；
其中位數 2.39% 低於 HDBSCAN 底的 2.46%。
若僅報告單次結果，將得出「Agglomerative 底的 DRL 表現最佳」之結論，
而該結論**無法在重訓下複現**。

跨策略比較一律以中位數為準；範圍寬度本身亦為一項結果指標
（反映該配對底提供的學習訊號穩定性）。

:::


## DRL 學到了什麼：決策行為解析

動作選擇未落庫，但可由 `trade_logs` 完整還原
（SKIP → 全期 `HOLD_CASH (SKIP)`；entry_z → 進場列的 $\min|Z|$）。
下表取每策略 TOP N 最大之網格（樣本最充足）。

**決策分布**（動作選單 $entry_z \in \{1.5, 2.0, 2.5, 3.0\}$，靜態基準 2.0）：

| 配對底 | 配對期數 | SKIP 率 | 門檻中位 | 1.5 / 2.0 / 2.5 / 3.0 | 偏離基準 |
| :--- | :---: | :---: | :---: | :---: | :---: |
| Agglomerative | 1462 | 35.0% | 2.22 | 12 / 40 / 21 / 27 % | **61%** |
| HDBSCAN | 1475 | 37.1% | 2.27 | 8 / 40 / 22 / 29 % | 62% |
| K-means | 1377 | 38.7% | 2.20 | 15 / 41 / 19 / 26 % | 62% |

**增益來源分解**（DRL − Z-Score 總損益差，逐配對拆解）：

| 配對底 | 總增益 | SKIP 貢獻 | 門檻貢獻 |
| :--- | :---: | :---: | :---: |
| Agglomerative | 3106 | 1117（36%） | **1989（64%）** |
| HDBSCAN | 845 | 260（31%） | **586（69%）** |
| K-means | 1361 | 711（52%） | 650（48%） |

::: {.callout-warning}

### SKIP 並非選擇性技巧

被 DRL 跳過的配對，其在 Z-Score 端的虧損比例與留下者**幾乎相同**
（Agglomerative 60% vs 60%；HDBSCAN 48% vs 51%；K-means 50% vs 48%），
Mann-Whitney 檢定 $p$ = 0.35 ~ 0.93，無一顯著。

「被跳過者損益為負」不足以證明技巧——配對平均損益本就為負，
**隨機棄權同樣會「避開虧損」**。判準須為其損益是否顯著低於留下者。

:::

**結論**：DRL 的價值在於**為每組配對挑選適當的進出場門檻**（62% 的決策偏離靜態基準），
而非拒絕交易。此結果在獨立資料上支持 Kim & Kim (2019) 的門檻最適化主張，
並量化了該機制的貢獻比重。

**可解釋性的邊界**：門檻選擇與排序名次的 Spearman $\rho$ 僅 −0.03 ~ +0.07，
模型並非依循「名次差則提高門檻」之類的簡單規則，
其決策為 12 維形成期特徵的非線性組合，本研究未能進一步歸因。


## 命題 2 的假設檢定（一）：相對主張的設計與逐輪複核

**$H_0$**：DRL 交易端與 Z-Score 交易端的績效相同。

檢定採**配對設計**——兩者跑在完全相同的配對、期間與參數格上，唯一差異為交易端。
市場崩盤、配對失效、成本衝擊等共同風險在相減後消去，
留下的差異序列僅含「交易端決策」一項變異。此即命題 2 檢定力遠高於絕對檢定的原因。

**逐輪複核**（三種 ML 配對底各五輪獨立重訓，各自檢定，
確認增益非特定訓練批次所致）：

| 配對底 | 各輪勝格數 | 各輪 $p$ 值範圍 |
| :--- | :--- | :---: |
| Agglomerative | 15/15　14/15　15/15　15/15　15/15 | 1.5e−4 ~ 1.0e−3 |
| HDBSCAN | 12/15　10/15　11/15　12/15　13/15 | 3.7e−3 ~ 1.2e−2 |
| K-means | 11/15　14/15　12/15　14/15　14/15 | 7.3e−4 ~ 4.8e−3 |

**15 個獨立檢定（3 配對底 × 5 輪）全數在 5% 水準下顯著**，
效果量均達 Cohen 之「大」標準（$d$ = 0.80 ~ 1.20）。

加計兩種 GICS 配對底的單輪檢定（$p$ = 0.0013 / 0.0070），
命題 2 於**五種配對底、共 17 個檢定中無一失敗**。


## 命題 2 的假設檢定（二）：絕對主張

**H0**：策略的平均日報酬為零。此處無對照組，須直接對抗市場噪音。

**Newey-West HAC $t$ 檢定**（Bartlett kernel，落後 10 階，6,287 交易日）：

| 配對底 | Z-Score | DRL |
| :--- | :---: | :---: |
| Agglomerative | $t$=1.16, $p$=0.248 | $t$=1.71, $p$=0.088 |
| HDBSCAN | $t$=1.02, $p$=0.309 | $t$=1.67, $p$=0.095 |
| K-means | $t$=0.86, $p$=0.391 | $t$=1.05, $p$=0.293 |

**Deflated Sharpe Ratio**（Bailey & López de Prado, 2014；校正 15 格網格搜尋的選擇偏誤）：

| 配對底 | Z-Score | DRL | DRL 門檻 $SR_0$ |
| :--- | :---: | :---: | :---: |
| Agglomerative | 0.047 | 0.561 | 0.313 |
| HDBSCAN | 0.002 | 0.467 | 0.325 |
| K-means | 0.020 | 0.329 | 0.292 |

**無一達到 0.95 判定門檻**。DRL 使 $p$ 值自 ~0.25 降至 0.088，方向一致但未跨越顯著水準。


## 兩項檢定為何結論相反

::: {.callout-important}

### 相對顯著、絕對不顯著，並不矛盾

兩者的**虛無假設不同**：

| | 配對 $t$ 檢定 | Deflated Sharpe |
| :--- | :--- | :--- |
| $H_0$ | 兩交易端績效相同 | 策略真實 Sharpe 為零 |
| 對照物 | 同配對、同參數格的 Z-Score | 零 |
| 主張性質 | 相對 | 絕對 |
| 校正對象 | —（配對設計消去共同風險） | 多重測試偏誤 |

配對設計消去了兩策略共同承受的市場風險，訊噪比大幅提高；
絕對檢定沒有這個對照，必須從市場噪音中直接辨識訊號。

:::

**檢定力受限於低 Sharpe，而非樣本長度。**
Sharpe 0.34、樣本 25 年，理論 $t \approx 0.34\sqrt{25} = 1.70$，與實測 1.71 幾乎相同。
欲使 Sharpe 0.34 之策略達到 $p < 0.05$，約需 33 年樣本。
此為成本後配對交易的普遍特性——Do & Faff (2010) 記錄了配對交易報酬自 2000 年代起的長期衰減。

**DSR 對比需保守解讀。**
DRL 的門檻 $SR_0$ 低於 Z-Score（0.31 vs 0.56），係因其網格間 Sharpe 變異較小。
此既反映真實的參數穩健性，亦會機械性推高 DSR，不宜將兩者落差全部歸因於績效改善。

> **本研究的檢定定位**：命題 2 為**方法之相對優劣**的假設檢定，證據充分；
> 絕對獲利能力則未達統計顯著，此點列於限制章。


## 命題 2 的 Regime 穩健性

以等權市場的 63 日滾動波動率三分位與 126 日趨勢標記每個交易日，
分層計算年化 Sharpe。檢驗增益是否僅來自特定市場環境。

| 配對底 / 交易端 | Calm | Normal | Turbulent | Bull | Bear |
| :--- | :---: | :---: | :---: | :---: | :---: |
| GICS × SSD | −0.72 | 0.03 | 0.51 | 0.02 | 0.35 |
| GICS × SSD **+ DRL** | **−0.47** | **0.14** | 0.51 | **0.10** | **0.41** |
| Agglomerative × SSD | −0.20 | −0.17 | 0.65 | 0.20 | 0.30 |
| Agglomerative **+ DRL** | **−0.07** | **−0.06** | **0.77** | **0.36** | **0.35** |
| HDBSCAN × SDP | −0.23 | −0.05 | 0.53 | 0.07 | 0.38 |
| HDBSCAN **+ DRL** | **−0.18** | **0.10** | **0.64** | **0.16** | **0.53** |
| K-means × SSD | −0.06 | −0.09 | 0.43 | 0.07 | 0.32 |
| K-means **+ DRL** | **−0.05** | **−0.05** | **0.47** | **0.13** | **0.34** |

**DRL 在 20 個 regime 格中改善 19 格、持平 1 格、劣化 0 格。**
命題 2 的增益不依賴特定市場環境。

**同時揭露一項策略層面的限制**：配對交易的獲利完全集中於
**高波動（0.43 ~ 0.77）與空頭（0.30 ~ 0.53）** 環境；
**平靜期所有策略皆為負 Sharpe**（−0.72 ~ −0.05），DRL 只能減輕虧損而無法反轉。
此與本研究另一項發現一致：損益集中於高分散度期間（附錄 D 的 regime 閘門依據）。


## 成本敏感度：Break-even 分析

成本模型可解析求解——進出場費用 = friction × 名目額，
且每配對名目額恰等於每配對資金，故往返 break-even
$c^* = 2\,(0.29\% + 	ext{淨利} / \Sigma	ext{名目額})$。

| 配對底 | Z-Score | → DRL | DRL 提升 |
| :--- | :---: | :---: | :---: |
| Agglomerative × SSD | 0.666% | **0.700%** | +0.034pp |
| HDBSCAN × SDP | 0.652% | **0.708%** | +0.056pp |
| GICS × SSD | 0.662% | **0.671%** | +0.009pp |
| K-means × SSD | 0.645% | **0.651%** | +0.006pp |

*往返 break-even 成本；現行假設為 0.58%（單邊 0.29%，Do & Faff 2012）*

**DRL 在四種配對底上皆提高成本承受度。** 機制為 SKIP 與較高的進場門檻
共同減少了交易次數，使同樣的毛利分攤在更少的名目額上。

::: {.callout-warning}

### 成本餘裕相當有限

最佳策略的往返 break-even 為 0.708%，僅較現行假設 0.58% 高出 **0.13pp（約 22%）**。
若實際摩擦成本高於文獻估計（如流動性較差的標的、較大的部位規模），
本研究的正報酬結論將不成立。

此與絕對顯著性檢定的結果一致：策略的邊際極薄，
**本研究之結論應限於方法間的相對比較**。

:::


# 附錄摘要

主軸之外的支撐性實驗，完整數據見 `results/result.db` 與
`archive/config_archived_strategies.py`。

| 附錄 | 內容 | 主要結果 |
| :--- | :--- | :--- |
| **A** | 文獻原始設定復現 | Gatev (2006) 原型、許鈞翔 (2025) ADF 0.01 設定；Grid (GICS-SSD) 與原生 SSD Rolling 數值完全相同 |
| **B** | 篩選消融（有/無三道統計過濾） | 篩選貢獻 +0.25 ~ +0.87pp，三種排序下皆為正 |
| **C** | 特徵工程消融 | 多尺度動量（−0.94 ~ −2.43pp）、SEC 結構性財報比率（±0.22pp 噪音範圍）皆無助益 |
| **D** | 延伸探索：regime 條件化進場 | 低分散度閘門使全網格 Sharpe 轉正、MDD 下降；三層疊加五輪中位 2.69%［2.65, 2.71］、最差輪 14/15 正 Sharpe |
| **I** | Regime／成本可重現腳本 | `analysis/regime_cost_dsr_eval.py`：regime 分層 Sharpe、break-even 成本、DSR |
| **H** | 粒度掃描可重現腳本 | `analysis/granularity_sweep.py`：切割門檻 × 候選池 × 跨產業比例 × 績效 |
| **G** | 行為解析可重現腳本 | `analysis/drl_behavior.py`：決策分布／SKIP 技巧性／增益來源分解 |
| **F** | 統計檢定可重現腳本 | `analysis/proposition2_stats.py`：配對 t／Wilcoxon／逐輪檢定／Newey-West／DSR 四項一次產出 |
| **E** | 參數敏感性 | ADF 門檻 0.01 / 0.05 / 0.1 對照，說明本研究採 0.05 的實證依據 |


## 附錄 B、E 的方法論意涵（值得在正文引用）

**篩選是必要的**：純距離排序不足以識別可交易配對——距離度量回答「歷史走勢多接近」，
共整合檢定回答「價差是否會回歸」，兩者結合才構成有效選取（SSD 排序下 0.79% → 1.66%）。

**ADF 門檻 0.05 優於文獻慣用的 0.01**：實證顯示過嚴門檻反而降低績效
（Agglomerative FMP：1.27% @0.01 vs 1.77% @0.05），且 0.05 → 0.1 已飽和。
機制為「排序流程先按距離排序、再逐一檢定並填滿 top_n」——門檻收緊迫使系統
往距離更遠的候選尋找，**以經濟相似性換取統計顯著性，淨效果為負**。
候選池充足時不再出現早期「篩選過嚴導致無配對」的問題（ADF 0.01 仍可填滿 20/20 對）。


# 結論與限制

## 結論

1. **命題 1 未獲支持**：機器學習分群未能建立優於 GICS 產業分類的配對搜尋空間。
   9 組配對檢定中 5 組顯著較差、0 組顯著較優；改用 DRL 交易端後結論不變
   （AGG vs GICS：$p = 0.016$，方向仍為 GICS 較優）。
   受控的粒度掃描進一步顯示：**分群演算法並非關鍵變因**——
   單一切割門檻參數造成的績效變動（2.05pp）遠大於三種演算法間的差距（0.48pp）。
   關係呈倒 U 形：粒度過細候選池不足，粒度過粗則跨產業配對自 10% 升至 50%。
   兩者在 ML 分群下無法解耦（$ho = +1.000$），而 GICS 同時具備
   「池大」與「產業純」兩項性質——此為其難以被超越的根本原因。

2. **命題 2 獲得支持**：DRL 交易端在**五種配對底**上皆顯著優於 Z-Score
   （$p$ = 0.0006 ~ 0.0070），且增益最大者為傳統 GICS 分組——
   此改良與配對來源正交，不依賴機器學習分群。

3. **增益具 regime 穩健性與成本穩健性**：DRL 在 20 個市場環境分層中改善 19 格、
   劣化 0 格；並在四種配對底上皆提高往返 break-even 成本（最高 0.708%）。

4. **增益機制可歸因**：損益分解顯示 48–69% 來自門檻選擇、31–52% 來自 SKIP，
   而 SKIP 的選擇性經檢定並不顯著。DRL 的實質貢獻為**情境化的門檻調校**。

5. **方法論觀察**：本研究有兩處結論在改用配對檢定後被推翻。
   以網格最佳值比較策略會系統性納入選擇偏誤，
   跨策略推論應以同參數格的配對檢定為準。

## 限制

- **策略之絕對獲利能力未達統計顯著**：Newey-West 檢定最佳 $p = 0.088$、
  Deflated Sharpe 最高 0.56（未達 0.95）。本研究之結論限於方法間的相對比較，
  不宣稱策略具可實現之超額報酬
- 命題 1 為「未獲支持」而非「GICS 顯著較優」：粒度掃描中各層級與 GICS 的
  配對檢定皆未達顯著（$p$ = 0.14 ~ 0.52）。結論限於本設定
  （混合特徵、S&P 500 大型股）。特徵工程消融（動量、結構性財報比率）
  皆為負面結果，但不排除其他特徵設計能表達產業以外的相似性維度而改變結論
- DRL 未固定隨機種子（已改以五輪中位數±範圍報告）；
  五輪對估計中位數尚屬有限，變異數本身的信賴區間未予量化
- DRL 的門檻選擇無法歸因至可解釋的簡單規則，模型內部決策依據仍為黑箱
- 網格最佳值存在多重測試偏誤，已以 Deflated Sharpe 量化其影響
- **成本餘裕極薄**：最佳策略往返 break-even 僅 0.708%，較現行假設 0.58% 高出 22%；
  摩擦成本若高於文獻估計，正報酬結論即不成立
- **獲利集中於特定市場環境**：平靜期所有策略皆為負 Sharpe（−0.72 ~ −0.05），
  獲利完全來自高波動與空頭環境；DRL 可減輕平靜期虧損但無法反轉
- 樣本限於 S&P 500 大型股，結論未必外推至中小型股或其他市場
